In [1]:
import sys
import os
import logging
import gc
import time
import torch
import warnings
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gliner import GLiNER
from gliner.data_processing.collator import DataCollator
from gliner.training import Trainer, TrainingArguments
from transformers import TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType,PeftModel

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
sys.path

/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python311.zip',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages',
 '/tmp/tmp6a_h5s9h']

In [2]:
os.path.dirname(os.getcwd())

'/opt/app/notebooks/abhishek/active_gliner'

In [3]:
src_path=os.path.join(os.path.dirname(os.getcwd()),'src')
sys.path.append(src_path)
sys.path


['/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python311.zip',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages',
 '/tmp/tmp6a_h5s9h',
 '/opt/app/notebooks/abhishek/active_gliner/src']

In [4]:


from config.settings import Settings
settings = Settings()

print(f"Settings cache_dir: {settings.cache_dir}")
print(f"Cache absolute path: {settings.cache_dir.resolve()}")
print(f"Does cache dir contain 'notebooks': {'notebooks' in str(settings.cache_dir)}")

Settings cache_dir: /opt/app/notebooks/abhishek/active_gliner/cache
Cache absolute path: /opt/app/notebooks/abhishek/active_gliner/cache
Does cache dir contain 'notebooks': True


In [5]:
# Cell 7: Integration test
print("=== Integration Test ===")
from utils.logging import setup_logging
from utils.reproducibility import set_all_seeds
from utils.device import setup_device

# Complete setup like your original code
settings = Settings()
settings.setup()  # Apply environment and create directories

logger = setup_logging(log_dir=str(settings.logs_dir))
set_all_seeds(seed=settings.global_seed, logger=logger)
device = setup_device(logger=logger)

logger.info("All modules integrated successfully!")
print(f"Final setup: seed={settings.global_seed}, device={device}, batch_size={settings.batch_size}")



INFO:ActiveLearning:================================================================================
INFO:ActiveLearning:ACTIVE LEARNING PIPELINE WITH PROPER TRAIN/TEST SEPARATION
INFO:ActiveLearning:================================================================================
INFO:ActiveLearning:Log file: /opt/app/notebooks/abhishek/active_gliner/logs/active_learning_20250910_190335.log
INFO:ActiveLearning:Setting all seeds to 42 for reproducibility...
INFO:ActiveLearning:Using device: cuda
INFO:ActiveLearning:CUDA version: 12.8
INFO:ActiveLearning:Number of GPUs visible: 1
INFO:ActiveLearning:Current GPU: 0
INFO:ActiveLearning:GPU Name: NVIDIA GeForce RTX 3090
INFO:ActiveLearning:GPU Memory: 23.6 GB
INFO:ActiveLearning:All modules integrated successfully!


=== Integration Test ===
Final setup: seed=42, device=cuda, batch_size=8


In [6]:
from data.loader import load_mit_dataset, load_dataset_from_config
from data.transforms import get_ner_statistics, log_ner_statistics  # Stats moved here
from config.settings import settings
from utils.logging import setup_logging

print("=== Testing Data Loader ===")

# Setup logger for testing
logger = setup_logging(log_dir="../logs", logger_name="DataTest")

# Test with dummy data (since actual files might not exist)

# Try to load real data if available
train_data_path = settings.data_path / settings.train_file
test_data_path = settings.data_path / settings.test_file
labels_path = settings.data_path / settings.labels_file

if test_data_path.exists() and labels_path.exists():
    train_data, entity_types = load_mit_dataset(str(train_data_path), str(labels_path), "test")
    print(f"Loaded train real data: {len(train_data)} examples, {len(entity_types)} entity types")
    test_data, entity_types = load_mit_dataset(str(test_data_path), str(labels_path), "test")
    print(f"Loaded test real data: {len(test_data)} examples, {len(entity_types)} entity types")
    

    # train unified stats function (no duplication now)
    stats = get_ner_statistics(train_data, entity_types)
    print(f"Dataset stats: {stats}")
    
    # Log stats using unified function
    log_ner_statistics(test_data, "train", logger, entity_types)



    # Test unified stats function (no duplication now)
    stats = get_ner_statistics(test_data, entity_types)
    print(f"Dataset stats: {stats}")
    
    # Log stats using unified function
    log_ner_statistics(test_data, "test", logger, entity_types)


INFO:DataTest:================================================================================
INFO:DataTest:ACTIVE LEARNING PIPELINE WITH PROPER TRAIN/TEST SEPARATION
INFO:DataTest:================================================================================
INFO:DataTest:Log file: ../logs/active_learning_20250910_190335.log


=== Testing Data Loader ===
Loading test data from: /opt/app/notebooks/abhishek/active_gliner/data/mit-movie/train.json
Processed 9774 examples
Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']
Loaded train real data: 9774 examples, 12 entity types
Loading test data from: /opt/app/notebooks/abhishek/active_gliner/data/mit-movie/test.json
Processed 2442 examples
Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']
Loaded test real data: 2442 examples, 12 entity types
Dataset stats: {'total_examples': 9774, 'avg_num_tokens': 10.17792101493759, 'avg_num_entities': 2.143953345610804, 'total_entities': 20955, 'unique_entity_types': 12, 'entity_type_counts': Counter({'genre': 4354, 'actor': 3220, 'year': 2858, 'title': 2376, 'plot': 1927, 'average ratings': 1866, 'director': 1720, 'rating': 1671, 'character': 384, '

INFO:DataTest:train Dataset Statistics:
INFO:DataTest:  Total examples: 2442
INFO:DataTest:  Avg num tokens: 10.10
INFO:DataTest:  Avg num entities: 2.15
INFO:DataTest:  Total entities: 5243
INFO:DataTest:  Unique entity types: 12
INFO:DataTest:  Top entity types: [('genre', 1117), ('actor', 812), ('year', 720), ('title', 561), ('plot', 491)]
INFO:DataTest:test Dataset Statistics:
INFO:DataTest:  Total examples: 2442
INFO:DataTest:  Avg num tokens: 10.10
INFO:DataTest:  Avg num entities: 2.15
INFO:DataTest:  Total entities: 5243
INFO:DataTest:  Unique entity types: 12
INFO:DataTest:  Top entity types: [('genre', 1117), ('actor', 812), ('year', 720), ('title', 561), ('plot', 491)]


Dataset stats: {'total_examples': 2442, 'avg_num_tokens': 10.104422604422604, 'avg_num_entities': 2.147010647010647, 'total_entities': 5243, 'unique_entity_types': 12, 'entity_type_counts': Counter({'genre': 1117, 'actor': 812, 'year': 720, 'title': 561, 'plot': 491, 'director': 456, 'average ratings': 449, 'rating': 408, 'character': 89, 'review': 56, 'song': 54, 'trailer': 30}), 'entity_type_coverage': {'genre': 1117, 'year': 720, 'plot': 491, 'average ratings': 449, 'actor': 812, 'title': 561, 'song': 54, 'character': 89, 'rating': 408, 'review': 56, 'director': 456, 'trailer': 30}}


In [7]:
from evaluation.evaluator import enhanced_evaluate, evaluate_and_extract_metrics

print("=== Testing Evaluation Evaluator ===")


from gliner import GLiNER
from gliner.data_processing.collator import DataCollator
from gliner.training import Trainer, TrainingArguments
from transformers import TrainerCallback

model = GLiNER.from_pretrained("knowledgator/modern-gliner-bi-large-v1.0")
model.config.max_len = 8192

if hasattr(model.data_processor, 'transformer_tokenizer'):    
    model.data_processor.transformer_tokenizer.model_max_length = 8192

# Get base parameter count
base_total = sum(p.numel() for p in model.model.parameters())
logger.info(f"Base Parameters: {base_total:,}")

model.eval()
model.to('cuda')
with torch.no_grad():
    test_results = enhanced_evaluate(
model,test_data,entity_types,threshold=0.5,batch_size=8,has_ground_truth=True,logger=logger
)


test_results

=== Testing Evaluation Evaluator ===


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 46892.84it/s]
INFO:DataTest:Base Parameters: 529,845,248
INFO:DataTest:Running enhanced evaluation...
INFO:DataTest:Processing 2442 examples...
INFO:DataTest:Analyzing errors with ground truth...


{'overall_metrics': {'total_predictions': 3635,
  'overall_confidence': np.float64(0.7866754065189597),
  'overall_confidence_pct': np.float64(78.66754065189598),
  'total_examples': 2442,
  'entity_level_accuracy': np.float64(0.3976730879267595),
  'entity_level_accuracy_pct': np.float64(39.76730879267595),
  'example_level_accuracy': 0.15192465192465193,
  'example_level_accuracy_pct': 15.192465192465193,
  'overall_f1': np.float64(0.46970038296913713),
  'overall_f1_pct': np.float64(46.970038296913714),
  'incorrect_examples': [{'tokenized_text': ['are',
     'there',
     'any',
     'good',
     'romantic',
     'comedies',
     'out',
     'right',
     'now'],
    'ner': [(4, 5, 'genre'), (7, 8, 'year')],
    'predictions': [[4, 5, 'genre']],
    'scores': [0.8240102529525757],
    'errors': {'false_negatives': [[7, 8, 'year']], 'false_positives': []}},
   {'tokenized_text': ['show',
     'me',
     'a',
     'movie',
     'about',
     'cars',
     'that',
     'talk'],
    'ne

In [8]:
from evaluation.helper import display_results

display_results(test_results)



ENHANCED EVALUATION RESULTS

Overall Metrics:
Total Predictions: 3,635
Overall Confidence: 0.7867 (78.67%)
Total Examples: 2,442
Example-Level Accuracy: 0.1519 (15.19%)
Entity-Level Accuracy: 0.3977 (39.77%)
Overall F1 Score: 0.4697 (46.97%)

Confidence Distribution:


,genre,year,plot,average ratings,actor,title,song,character,rating,review,director,trailer
Confidence Range,,,,,,,,,,,,
0-25%,0,0,0,0,0,0,0,0,0,0,0,0
26-50%,0,0,0,0,0,0,0,0,0,0,0,0
51-75%,352,82,219,4,153,15,11,120,171,45,101,18
76-100%,478,196,318,5,468,9,24,163,342,32,186,1



Classification Report:


,entity_type,tp,fp,fn,precision,recall,f1,support,avg_prediction_confidence
0,genre,562,297,555,0.654249,0.503133,0.568826,1117,0.77
1,year,205,83,515,0.711806,0.284722,0.406746,720,0.81
2,plot,138,412,353,0.250909,0.281059,0.265130,491,0.77
3,actor,585,52,227,0.918367,0.720443,0.807453,812,0.82
4,average ratings,1,8,448,0.111111,0.002227,0.004367,449,0.72
5,character,67,230,22,0.225589,0.752809,0.347150,89,0.78
6,title,2,27,559,0.068966,0.003565,0.006780,561,0.68
7,song,20,16,34,0.555556,0.370370,0.444444,54,0.80
8,rating,205,328,203,0.384615,0.502451,0.435707,408,0.79
9,review,6,77,50,0.072289,0.107143,0.086331,56,0.73



True Positives Confidence Analysis:


,genre,year,plot,average ratings,actor,title,song,character,rating,review,director,trailer
Confidence Range,,,,,,,,,,,,
0-25%,0,0,0,0,0,0,0,0,0,0,0,0
26-50%,0,0,0,0,0,0,0,0,0,0,0,0
51-75%,163,22,63,1,136,2,3,10,72,1,98,4
76-100%,388,179,73,0,437,0,17,56,123,4,184,0



False Positives Confidence Analysis:


,genre,year,plot,average ratings,actor,title,song,character,rating,review,director,trailer
Confidence Range,,,,,,,,,,,,
0-25%,0,0,0,0,0,0,0,0,0,0,0,0
26-50%,0,0,0,0,0,0,0,0,0,0,0,0
51-75%,189,60,156,3,17,13,8,110,99,44,3,14
76-100%,90,17,245,5,31,9,7,107,219,28,2,1



Incorrect Examples: 2071
Corrected Labels Available: 2071


In [9]:
from data.loader import load_json_file
from selection.strategies import get_lowest_score_examples_sorted


model.eval()
model.to('cuda')
with torch.no_grad():
    train_results=enhanced_evaluate(model,train_data,entity_types,threshold=0.5,batch_size=8,has_ground_truth=True,logger=logger)



print(train_results['overall_metrics']['overall_f1_pct'])

low_n=get_lowest_score_examples_sorted(train_results,1000,logger=logger)

print(len(low_n))



INFO:DataTest:Running enhanced evaluation...
INFO:DataTest:Processing 9774 examples...
INFO:DataTest:Analyzing errors with ground truth...
INFO:DataTest:Extracting 1000 lowest confidence examples from training pool evaluation results...
INFO:DataTest:Total examples available: 9774
INFO:DataTest:Returning 1000 lowest confidence examples
INFO:DataTest:  Example 1: score=0.500, text='johnny depp made what movies in the 1980s...'
INFO:DataTest:  Example 2: score=0.500, text='what is the veiwers rating on like mike...'
INFO:DataTest:  Example 3: score=0.500, text='can give give me the critically acclaimed comedies...'


47.78185690665022
1000


In [10]:
import json
file_path="../results/low_score_1000_examples.json"
with open(file_path, 'w') as f:
    json.dump(low_n, f, indent=2)
print(f"Saved data to {file_path}")

Saved data to ../results/low_score_1000_examples.json
